# Kalman Filter（卡爾曼濾波，1960）

- **核心原理**：狀態空間的預測＋校正
- **變體**：Kalman smoother（RTS）、Extended KF（非線性，線性化）、Unscented KF（非線性，sigma points）、粒子濾波
- **推薦資料集**：自己模擬的帶雜訊軌跡（NumPy 生成）—— **知道真相**，所以每一步都可以檢查
- **文獻**：Kalman (1960), *A New Approach to Linear Filtering and Prediction Problems*；Rauch, Tung & Striebel (1965)；Durbin & Koopman, *Time Series Analysis by State Space Methods*

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

前幾章的模型其實都可以寫成**狀態空間模型**：

$$\underbrace{\mathbf x_t = F\mathbf x_{t-1} + \mathbf w_t,\ \ \mathbf w_t \sim N(0, Q)}_{\text{狀態方程：看不到的真實狀態怎麼演變}} \qquad \underbrace{\mathbf y_t = H\mathbf x_t + \mathbf v_t,\ \ \mathbf v_t \sim N(0, R)}_{\text{觀測方程：看得到的量測 = 狀態 + 雜訊}}$$

| | 前幾章 | 本章 |
| --- | --- | --- |
| 狀態 | 指數平滑的 $\ell_t, b_t, s_t$；ARIMA 的過去值與誤差；HMM 的離散狀態（第一部分第 20 章） | 一般的連續向量 $\mathbf x_t$（例如位置 + 速度） |
| 怎麼更新 | 各章各自的遞迴公式 | **一套通用的遞迴**：預測（用模型往前推）→ 校正（用量測拉回來），權重（Kalman gain）**自動**由不確定性決定 |
| 不確定性 | 大多只有點估計 | 每一步都有**共變異數** $P_t$：知道自己有多不確定 |

一句話：**Kalman filter = 每一步都在「模型的預測」和「雜訊的量測」之間做加權平均，權重看誰比較可信（變異數比較小）—— 對線性、高斯的模型，這就是精確的貝氏後驗。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：本章自己模擬兩種資料（`simulate_cv`、`simulate_local_level`，第 1 節）：

1. **2D 等速運動**（主角）：物體在平面上移動，狀態 $\mathbf x = (p_x, p_y, v_x, v_y)$（位置、速度），每一步速度被隨機的加速度推一下；我們只量得到**有雜訊的位置**。
2. **局部水準模型**（1D）：$\mu_t = \mu_{t-1} + \eta_t$、$y_t = \mu_t + \varepsilon_t$ —— 用來跟第 1 章的指數平滑、statsmodels 對照。

（`learning_map.data.simulate_trajectory()` 也可以產生軌跡；本章直接寫出模擬的程式，因為 $F, Q, H, R$ 是什麼、是本章要學的東西。）

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 2D 等速運動的值 |
| --- | --- | --- |
| $\mathbf x_t$、$\mathbf y_t$ | 狀態、量測 | $(4,)$、$(2,)$ |
| $F$、$H$ | 狀態轉移矩陣、觀測矩陣 | $(4, 4)$、$(2, 4)$ |
| $Q$、$R$ | 過程雜訊、量測雜訊的共變異數 | $(4, 4)$、$(2, 2)$ |
| $\hat{\mathbf x}_{t\mid t-1}$、$P_{t\mid t-1}$ | **預測**：看到 $\mathbf y_t$ 之前的估計與共變異數 | |
| $\hat{\mathbf x}_{t\mid t}$、$P_{t\mid t}$ | **濾波**：看到 $\mathbf y_t$ 之後 | |
| $\hat{\mathbf x}_{t\mid n}$ | **平滑**：用全部 $n$ 個量測（含未來）的估計 | |
| $\mathbf v_t$、$S_t$ | 創新（innovation）$\mathbf y_t - H\hat{\mathbf x}_{t\mid t-1}$ 與它的共變異數 | $(2,)$、$(2, 2)$ |
| $K_t$ | Kalman gain | $(4, 2)$ |
| $q$、$r$ | 加速度雜訊、量測雜訊的大小（純量） | |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 1 章 | 指數平滑 | SES：$\ell_t = \ell_{t-1} + \alpha(y_t - \ell_{t-1})$，$\alpha$ 固定。本章會看到：局部水準模型的 Kalman gain 收斂到一個常數 = SES 的 $\alpha$。 |
| 第 2、4 章 | ARIMA、GARCH | 最大概似估計（數值最佳化）；GARCH 用的是「一步預測誤差」的常態 log-likelihood —— Kalman filter 的 log-likelihood 也是用一步預測誤差（創新）算。 |
| 第一部分第 20 章 | HMM | forward 演算法：$\alpha_t = (\alpha_{t-1}A)\odot B_{\cdot,x_t}$ —— 「先預測、再用觀測修正」。Kalman filter 是**連續、高斯**版的 forward；RTS smoother 是 forward-backward 的 backward。 |
| 第一部分第 15 章 | GMM | 多變量常態、共變異數矩陣；常態的條件分布仍是常態。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 兩個估計的加權平均**：同一個量有兩個獨立的估計 $a$（變異 $\sigma_a^2$）與 $b$（變異 $\sigma_b^2$），最好的組合是 $a + K(b - a)$，$K = \frac{\sigma_a^2}{\sigma_a^2 + \sigma_b^2}$；組合後的變異 $(1 - K)\sigma_a^2$ 比兩者都小。例：$a = 0$（變異 2）、$b = 2$（變異 1）→ $K = 2/3$、組合 $= 4/3$、變異 $2/3$。這就是一維的 Kalman 更新。

**2. 線性轉換的共變異數**：$\mathbf z = A\mathbf x$ → $\text{Cov}(\mathbf z) = A\,\text{Cov}(\mathbf x)\,A^\top$；兩個獨立的量相加，共變異數相加。

**3. 等速運動的離散化**：時間步 $\Delta t$，位置 $p_t = p_{t-1} + v_{t-1}\Delta t + \frac12 a\Delta t^2$、速度 $v_t = v_{t-1} + a\Delta t$；$a$ 是隨機加速度（變異 $q$）→ $Q = q\,GG^\top$，$G = (\frac12\Delta t^2, \Delta t)^\top$（每個軸）。

**4. 多變量常態的條件分布**：$\begin{pmatrix}\mathbf x\\\mathbf y\end{pmatrix} \sim N\Big(\begin{pmatrix}\mu_x\\\mu_y\end{pmatrix}, \begin{pmatrix}\Sigma_{xx} & \Sigma_{xy}\\\Sigma_{yx} & \Sigma_{yy}\end{pmatrix}\Big)$ → $E[\mathbf x\mid\mathbf y] = \mu_x + \Sigma_{xy}\Sigma_{yy}^{-1}(\mathbf y - \mu_y)$、$\text{Cov}[\mathbf x\mid\mathbf y] = \Sigma_{xx} - \Sigma_{xy}\Sigma_{yy}^{-1}\Sigma_{yx}$。Kalman 更新就是這個公式（$\Sigma_{xy} = PH^\top$、$\Sigma_{yy} = HPH^\top + R$）。

**5. 卡方分布**：$\mathbf v \sim N(0, S)$（$d$ 維）→ $\mathbf v^\top S^{-1}\mathbf v \sim \chi^2_d$，平均 $= d$。（第 7 節的 NIS 檢查。）

#### 程式

**6. `@` 與 `.T`**；`np.linalg.solve(S, b)`（比 `inv(S) @ b` 穩定）；`np.linalg.slogdet(S)` → $\log\det S$。
**7. `np.isnan(y).any()`**：量測缺漏時跳過更新。

---
## 1. 資料（模擬）與探索性資料分析

**2D 等速運動**：$\Delta t = 1$、加速度雜訊 $q = 0.05$、量測雜訊標準差 $\sqrt r = 2$（$r = 4$）、200 步。產生三條：**train**（seed 0，估計／調參）、**validation**（seed 1）、**test**（seed 2，最後才看）。因為是模擬，**真實狀態都知道** —— 評估直接看「估計的位置／速度離真相多遠」（RMSE）。

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
DT, Q_TRUE, R_TRUE, N_STEPS = 1.0, 0.05, 4.0, 200


def cv_matrices(q: float, r: float, dt: float = DT) -> dict:
    """2D 等速運動的 F、H、Q、R。狀態 (px, py, vx, vy)；只量位置。"""
    F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=float)
    G = np.array([[dt**2 / 2, 0], [0, dt**2 / 2], [dt, 0], [0, dt]])
    H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=float)
    return {"F": F, "H": H, "Q": q * G @ G.T, "R": r * np.eye(2)}


def simulate_cv(n: int = N_STEPS, q: float = Q_TRUE, r: float = R_TRUE, seed: int = 0, x0: np.ndarray | None = None) -> dict:
    """回傳 {"x": 真實狀態 (n, 4)，"y": 量測 (n, 2)}。"""
    rng = np.random.default_rng(seed)
    G = np.array([[DT**2 / 2, 0], [0, DT**2 / 2], [DT, 0], [0, DT]])
    m = cv_matrices(q, r)
    x = np.array([0.0, 0.0, 1.0, 0.5]) if x0 is None else x0
    xs, ys = [], []
    for _ in range(n):
        x = m["F"] @ x + G @ rng.normal(0, np.sqrt(q), 2)
        xs.append(x)
        ys.append(m["H"] @ x + rng.normal(0, np.sqrt(r), 2))
    return {"x": np.array(xs), "y": np.array(ys)}


sim_tr, sim_va, sim_te = simulate_cv(seed=0), simulate_cv(seed=1), simulate_cv(seed=2)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(sim_tr["x"][:, 0], sim_tr["x"][:, 1], color="k", label="true path")
axes[0].scatter(sim_tr["y"][:, 0], sim_tr["y"][:, 1], s=6, color="tab:red", alpha=0.5, label="measurements")
axes[0].legend()
axes[0].set_aspect("equal")
axes[0].set_title("train trajectory (seed 0)")
axes[1].plot(sim_tr["x"][:, 2], label="true vx")
axes[1].plot(np.diff(sim_tr["y"][:, 0], prepend=np.nan), lw=0.5, alpha=0.6, label="naive vx = Δ measured px")
axes[1].legend()
axes[1].set_title("velocity: true vs finite difference of measurements")
plt.show()
err = sim_tr["y"] - sim_tr["x"][:, :2]
print("量測誤差的標準差（應 ≈ √r = 2）：", err.std(axis=0).round(3), "；用相鄰量測相減估計速度的 RMSE：",
      np.sqrt(np.nanmean((np.diff(sim_tr["y"][:, 0]) - sim_tr["x"][1:, 2]) ** 2)).round(3), "（真實 vx 的標準差", sim_tr["x"][:, 2].std().round(3), "）")

In [ ]:
# 局部水準模型（1D）：μ_t = μ_{t−1} + η_t（變異 q_ll）、y_t = μ_t + ε_t（變異 r_ll）
def simulate_local_level(n: int = 300, q: float = 0.5, r: float = 2.0, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    mu = np.cumsum(rng.normal(0, np.sqrt(q), n))
    return {"x": mu, "y": mu + rng.normal(0, np.sqrt(r), n)}


ll_sim = simulate_local_level()
fig, ax = plt.subplots(figsize=(14, 3.2))
ax.plot(ll_sim["y"], lw=0.6, color="tab:red", label="y (observed)")
ax.plot(ll_sim["x"], color="k", label="μ (true level)")
ax.legend()
plt.show()

In [ ]:
# TODO 1（問答）資料小結
#
# (1) 量測誤差的標準差是多少？跟設定的 √r = 2 一致嗎？
# (2) 「相鄰兩個量測相減」當速度：RMSE 是多少？比真實速度本身的標準差大還是小？為什麼這麼差？
#     （提示：Δy = Δx + (ε_t − ε_{t−1})，雜訊的變異變成 2r = 8）
# (3) 狀態裡哪些量「看得到」、哪些「看不到」？我們沒有任何速度的量測，卻要估計速度 —— 資訊從哪裡來？（提示：F 把位置和速度綁在一起）
# 答：

---
## 2. Baseline（在 validation 軌跡上，只用到「現在以前」的量測）

| Baseline | 位置估計 | 速度估計 |
| --- | --- | --- |
| 原始量測 | $\mathbf y_t$ | 相鄰差分 $\mathbf y_t - \mathbf y_{t-1}$ |
| 移動平均（過去 5 步） | 平均 | 5 步的平均斜率（最小平方的直線） |
| 指數平滑 | 位置用 SES（$\alpha = 0.3$） | 對差分做 SES |

**指標**：位置 RMSE、速度 RMSE（跟真相比；前 10 步不算，讓每個方法都「暖機」）。

In [ ]:
BURN = 10


def rmse_state(est_pos: np.ndarray, est_vel: np.ndarray, truth: np.ndarray) -> dict:
    return {"pos RMSE": float(np.sqrt(np.nanmean((est_pos[BURN:] - truth[BURN:, :2]) ** 2))),
            "vel RMSE": float(np.sqrt(np.nanmean((est_vel[BURN:] - truth[BURN:, 2:]) ** 2)))}


def ses(y: np.ndarray, alpha: float) -> np.ndarray:
    out = np.empty_like(y)
    out[0] = y[0]
    for t in range(1, len(y)):
        out[t] = out[t - 1] + alpha * (y[t] - out[t - 1])
    return out


def moving_line(y: np.ndarray, w: int = 5) -> tuple[np.ndarray, np.ndarray]:
    """過去 w 步做直線回歸：回傳 (目前位置的估計, 斜率)。"""
    pos, vel = np.full_like(y, np.nan), np.full_like(y, np.nan)
    tt = np.arange(w) - (w - 1)
    for t in range(w - 1, len(y)):
        for d in range(y.shape[1]):
            b1, b0 = np.polyfit(tt, y[t - w + 1 : t + 1, d], 1)
            pos[t, d], vel[t, d] = b0, b1
    return pos, vel


yv, xv = sim_va["y"], sim_va["x"]
diff_v = np.vstack([np.full((1, 2), np.nan), np.diff(yv, axis=0)])
base = {
    "raw measurement": rmse_state(yv, diff_v, xv),
    "moving line (5)": rmse_state(*moving_line(yv, 5), xv),
    "SES α=0.3": rmse_state(np.column_stack([ses(yv[:, d], 0.3) for d in range(2)]), np.column_stack([ses(np.nan_to_num(diff_v[:, d]), 0.3) for d in range(2)]), xv),
}
val_table = pd.DataFrame(base).T
display(val_table.round(4))

In [ ]:
# TODO 2（問答）
#
# (1) 原始量測的位置 RMSE 大約是多少？為什麼剛好接近 √r？
# (2) 移動平均、SES 讓位置變好還是變差？為什麼「平均過去的位置」在物體一直移動時會有偏差（落後）？
# (3) 先寫下預測：Kalman filter 的位置 RMSE 會比原始量測好多少？速度呢？
# 答：

---
## 3. 手寫模型：Kalman filter 與 RTS smoother

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 預測步 | 3 |
| 3B | 校正步（更新）；跳過缺漏的量測 | 4 |
| 3C | 創新的 log-likelihood | 5 |
| 3D | RTS smoother（用未來的量測回頭修正） | 6 |
| 3E | 檢查：暴力的高斯條件分布、statsmodels；實驗 | 7、8 |

**程式結構**：`KalmanFilter(F, H, Q, R, x0, P0)` —— `predict`、`update` 是一步；`filter(Y)` 走過整條序列，回傳 dict（`xp`、`Pp`：預測；`xf`、`Pf`：濾波；`v`、`S`：創新；`loglik`）；`smooth(out)` 從 `filter` 的結果倒著走。

### 3A. 預測步

$$\hat{\mathbf x}_{t\mid t-1} = F\hat{\mathbf x}_{t-1\mid t-1}, \qquad P_{t\mid t-1} = F P_{t-1\mid t-1}F^\top + Q$$

用模型往前推一步：估計跟著動、不確定性也跟著傳（前置複習 2），再加上這一步新的過程雜訊 $Q$。

### 3B. 校正步

$$\mathbf v_t = \mathbf y_t - H\hat{\mathbf x}_{t\mid t-1},\quad S_t = HP_{t\mid t-1}H^\top + R,\quad K_t = P_{t\mid t-1}H^\top S_t^{-1}$$
$$\hat{\mathbf x}_{t\mid t} = \hat{\mathbf x}_{t\mid t-1} + K_t\mathbf v_t, \qquad P_{t\mid t} = (I - K_tH)\,P_{t\mid t-1}$$

讀法：$\mathbf v_t$ = 量測跟預測差多少（意外）；$S_t$ = 意外「應該」有多大；$K_t$ = 意外要採信多少（$P$ 大、$R$ 小 → 多採信量測）。一維時就是前置複習 1。

**缺漏的量測**（$\mathbf y_t$ 有 NaN）：跳過校正，$\hat{\mathbf x}_{t\mid t} = \hat{\mathbf x}_{t\mid t-1}$、$P_{t\mid t} = P_{t\mid t-1}$（不確定性只增不減）。

### 3C. log-likelihood

一步預測的分布 $\mathbf y_t \mid \mathbf y_{1..t-1} \sim N(H\hat{\mathbf x}_{t\mid t-1}, S_t)$ →
$$\log L = -\frac12\sum_t\Big[d\log 2\pi + \log\det S_t + \mathbf v_t^\top S_t^{-1}\mathbf v_t\Big]$$
（$d$ = 量測的維度）—— 用來估計 $q$、$r$（第 4 節），跟第 4 章 GARCH 的概似是同一個想法。

#### 符號表（3A–3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\hat{\mathbf x}$、$P$ | 上一步濾波後的估計、共變異數 | $(k,)$、$(k, k)$ | `x`、`P` |
| $\hat{\mathbf x}_{t\mid t-1}$、$P_{t\mid t-1}$ | 預測 | 同上 | `x_pred`、`P_pred` |
| $\mathbf v$、$S$、$K$ | 創新、創新共變異數、gain | $(d,)$、$(d, d)$、$(k, d)$ | `v`、`S`、`K` |

**下標公式**：
- TODO 3：`x_pred = F @ x`；`P_pred = F @ P @ F.T + Q`。
- TODO 4：`v = y − H @ x_pred`；`S = H @ P_pred @ H.T + R`；`K = P_pred @ H.T @ np.linalg.inv(S)`（或 `np.linalg.solve(S, H @ P_pred).T`）；`x_new = x_pred + K @ v`；`P_new = (I − K @ H) @ P_pred`。
- TODO 5：每一步 `ll += −0.5 * (d * log(2π) + slogdet(S)[1] + v @ solve(S, v))`。

#### 最小例子：TODO 3、4、5（一維）

$F = H = 1$、$Q = R = 1$、$\hat x_0 = 0$、$P_0 = 1$，量測 $y_1 = 2$。

1. **正確答案**（手算，前置複習 1）：預測 $\hat x_{1\mid 0} = 0$、$P_{1\mid 0} = 2$；$v = 2$、$S = 3$、$K = 2/3$；$\hat x_{1\mid 1} = 4/3 \approx 1.3333$、$P_{1\mid 1} = 2/3 \approx 0.6667$；log-likelihood $= -\frac12[\log 2\pi + \log 3 + 4/3] \approx -2.1349$。
2. **積木**：一維時 `F @ x` 就是 `x`，矩陣版本的 shape 要是 $(1,)$、$(1, 1)$。
3. **留給你**：TODO 3、4、5。

In [ ]:
# 最小例子：TODO 3、4、5（一維）—— 1. 正確答案：純量
x_s, P_s, y_s = 0.0, 1.0, 2.0
xp_s, Pp_s = x_s, P_s + 1.0
S_s = Pp_s + 1.0
K_s = Pp_s / S_s
print(f"預測 {xp_s}、{Pp_s}；S = {S_s}、K = {K_s:.4f}；濾波 {xp_s + K_s * (y_s - xp_s):.4f}、{(1 - K_s) * Pp_s:.4f}；"
      f"log-lik = {-0.5 * (np.log(2 * np.pi) + np.log(S_s) + (y_s - xp_s) ** 2 / S_s):.4f}")
# 寫完 TODO 3–5、執行過下一格之後，取消下面的註解來對照：
# kf_mini = KalmanFilter(np.eye(1), np.eye(1), np.eye(1), np.eye(1), np.zeros(1), np.eye(1)); o_mini = kf_mini.filter(np.array([[2.0]])); print(o_mini["xf"].round(4), o_mini["Pf"].round(4), round(o_mini["loglik"], 4))

In [ ]:
from typing import Self


class KalmanFilter:
    """線性高斯狀態空間模型的 Kalman filter 與 RTS smoother。"""

    def __init__(self, F: np.ndarray, H: np.ndarray, Q: np.ndarray, R: np.ndarray, x0: np.ndarray, P0: np.ndarray) -> None:
        self.F, self.H, self.Q, self.R, self.x0, self.P0 = F, H, Q, R, x0, P0

    def predict(self, x: np.ndarray, P: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        # TODO 3（程式）：預測步（兩行）
        raise NotImplementedError("TODO 3：預測")
        return x_pred, P_pred

    def update(self, x_pred: np.ndarray, P_pred: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """回傳 (x_new, P_new, v, S)。y 有 NaN 時跳過校正（v、S 回傳 NaN）。"""
        if np.isnan(y).any():
            d = len(y)
            return x_pred, P_pred, np.full(d, np.nan), np.full((d, d), np.nan)
        H, R = self.H, self.R
        # TODO 4（程式）：v、S、K、x_new、P_new（見下標公式）
        raise NotImplementedError("TODO 4：校正")
        return x_new, P_new, v, S

    def filter(self, Y: np.ndarray) -> dict:
        """Y (n, d)。回傳 xp、Pp、xf、Pf (n, ...)、v (n, d)、S (n, d, d)、loglik。"""
        n, d = Y.shape
        x, P = self.x0, self.P0
        out = {k: [] for k in ["xp", "Pp", "xf", "Pf", "v", "S"]}
        ll = 0.0
        for t in range(n):
            x_pred, P_pred = self.predict(x, P)
            x, P, v, S = self.update(x_pred, P_pred, Y[t])
            if not np.isnan(v).any():
                # TODO 5（程式）：ll 加上這一步的 log-likelihood（見下標公式；用 np.linalg.slogdet、np.linalg.solve）
                raise NotImplementedError("TODO 5：log-likelihood")
            for k, val in zip(["xp", "Pp", "xf", "Pf", "v", "S"], [x_pred, P_pred, x, P, v, S]):
                out[k].append(val)
        out = {k: np.array(v_) for k, v_ in out.items()}
        out["loglik"] = float(ll)
        return out

    def smooth(self, out: dict) -> tuple[np.ndarray, np.ndarray]:
        """RTS smoother：回傳 (xs (n, k)、Ps (n, k, k))。"""
        xf, Pf, xp, Pp = out["xf"], out["Pf"], out["xp"], out["Pp"]
        n = len(xf)
        xs, Ps = xf.copy(), Pf.copy()
        for t in range(n - 2, -1, -1):
            # TODO 6（程式）：C = Pf[t] @ F.T @ inv(Pp[t + 1])；xs[t] = xf[t] + C @ (xs[t + 1] − xp[t + 1])；Ps[t] = Pf[t] + C @ (Ps[t + 1] − Pp[t + 1]) @ C.T
            raise NotImplementedError("TODO 6：RTS smoother")
        return xs, Ps

### 3D. RTS smoother

濾波 $\hat{\mathbf x}_{t\mid t}$ 只用到「現在以前」的量測（即時追蹤用）；事後分析時，**未來的量測也能告訴我們過去在哪裡**。RTS smoother 從最後一步倒著走：

$$C_t = P_{t\mid t}F^\top P_{t+1\mid t}^{-1}, \quad \hat{\mathbf x}_{t\mid n} = \hat{\mathbf x}_{t\mid t} + C_t(\hat{\mathbf x}_{t+1\mid n} - \hat{\mathbf x}_{t+1\mid t}), \quad P_{t\mid n} = P_{t\mid t} + C_t(P_{t+1\mid n} - P_{t+1\mid t})C_t^\top$$

讀法：「下一步的平滑估計」跟「下一步的預測」差多少（未來告訴我們的新資訊），用 $C_t$ 傳回這一步。（第一部分 HMM 的 backward 也是同一個角色。）

### 3E. 檢查與實驗

**檢查 1：暴力的高斯條件分布**。狀態與量測全部疊起來是一個大的多變量常態（前置複習 4）。$T = 6$ 步時（$\mathbf X$ 是 24 維、$\mathbf Y$ 是 12 維），直接用 $E[\mathbf X\mid\mathbf Y]$ 的公式算 —— 應該跟 **smoother** 完全一樣；只用前 $t$ 個量測時，應該跟 **filter** 的第 $t$ 步一樣。

**檢查 2：statsmodels**。局部水準模型 = statsmodels 的 `UnobservedComponents(level="llevel")`。給定同樣的 $q, r$ 與初始值，濾波狀態與每一步的 log-likelihood 應該一樣。（statsmodels 的 `initialize_known` 設定的是「第一步的預測」$P_{1\mid 0}$，所以要傳 $P_0 + q$；它的 `llf` 會扣掉第一步，所以比 `llf_obs`。）

In [ ]:
# 檢查 1：暴力的高斯條件分布（T = 6）
m4 = cv_matrices(Q_TRUE, R_TRUE)
x0_, P0_ = np.zeros(4), np.eye(4) * 10
T_small = 6
Y_small = sim_tr["y"][:T_small]
F_, Q_, H_, R_ = m4["F"], m4["Q"], m4["H"], m4["R"]
Fpow = [np.linalg.matrix_power(F_, i) for i in range(T_small + 1)]
mean_X = np.concatenate([Fpow[t + 1] @ x0_ for t in range(T_small)])
Cxx = np.zeros((4 * T_small, 4 * T_small))
for i in range(T_small):
    for j in range(T_small):
        c = Fpow[i + 1] @ P0_ @ Fpow[j + 1].T + sum(Fpow[i - s] @ Q_ @ Fpow[j - s].T for s in range(min(i, j) + 1))
        Cxx[4 * i : 4 * i + 4, 4 * j : 4 * j + 4] = c


def brute_posterior_mean(t_obs: int) -> np.ndarray:
    """用前 t_obs 個量測，所有 T_small 步狀態的後驗平均 (T_small, 4)。"""
    Hb = np.kron(np.eye(T_small), H_)[: 2 * t_obs]
    Cyy = Hb @ Cxx @ Hb.T + np.kron(np.eye(t_obs), R_)
    return (mean_X + Cxx @ Hb.T @ np.linalg.solve(Cyy, Y_small[:t_obs].ravel() - Hb @ mean_X)).reshape(T_small, 4)


kf_small = KalmanFilter(F_, H_, Q_, R_, x0_, P0_)
o_small = kf_small.filter(Y_small)
xs_small, _ = kf_small.smooth(o_small)
print(f"✅ smoother vs 暴力（全部 6 個量測）：{np.abs(xs_small - brute_posterior_mean(T_small)).max():.1e}")
print(f"✅ filter 第 t 步 vs 暴力（前 t 個量測）：{max(np.abs(o_small['xf'][t - 1] - brute_posterior_mean(t)[t - 1]).max() for t in range(1, T_small + 1)):.1e}")

In [ ]:
# 檢查 2：局部水準模型 vs statsmodels UnobservedComponents
from statsmodels.tsa.statespace.structural import UnobservedComponents

q_ll, r_ll, P0_ll = 0.5, 2.0, 10.0
kf_ll = KalmanFilter(np.eye(1), np.eye(1), np.eye(1) * q_ll, np.eye(1) * r_ll, np.zeros(1), np.eye(1) * P0_ll)
o_ll = kf_ll.filter(ll_sim["y"][:, None])
uc = UnobservedComponents(ll_sim["y"], level="llevel")
uc.initialize_known(np.array([0.0]), np.array([[P0_ll + q_ll]]))
uc_res = uc.smooth([r_ll, q_ll])  # 參數順序：sigma2.irregular（r）、sigma2.level（q）
xs_ll, _ = kf_ll.smooth(o_ll)
print(f"✅ 濾波狀態差 {np.abs(o_ll['xf'][:, 0] - uc_res.filtered_state[0]).max():.1e}；平滑狀態差 {np.abs(xs_ll[:, 0] - uc_res.smoothed_state[0]).max():.1e}；"
      f"log-lik 總和差（用 llf_obs）{abs(o_ll['loglik'] - uc_res.llf_obs.sum()):.1e}")

In [ ]:
# 實驗 1：Kalman gain 收斂到常數 —— 局部水準模型的穩態 gain = 第 1 章 SES 的 α
K_path = np.array([P[0, 0] / (P[0, 0] + r_ll) for P in o_ll["Pp"]])
ratio = q_ll / r_ll
K_theory = (-ratio + np.sqrt(ratio**2 + 4 * ratio)) / 2  # 穩態的 P/(P + r)，解 Riccati 方程式
print(f"Kalman gain 前 5 步：{K_path[:5].round(4)}；第 50 步：{K_path[50]:.6f}；穩態公式：{K_theory:.6f}")
print(f"→ 用 α = {K_theory:.4f} 的 SES 跟 Kalman 濾波的最大差異（第 50 步以後；兩者初始值不同，但 50 步後初始值的影響已經消失）：",
      f"{np.abs(ses(ll_sim['y'], K_theory)[50:] - o_ll['xf'][50:, 0]).max():.2e}")

# 實驗 2：q/r 比例（2D，validation 軌跡）：相信模型 vs 相信量測
rows = []
for q_try in [0.0005, 0.005, 0.05, 0.5, 5.0]:
    m = cv_matrices(q_try, R_TRUE)
    o = KalmanFilter(m["F"], m["H"], m["Q"], m["R"], np.zeros(4), np.eye(4) * 100).filter(yv)
    rows.append({"q (filter assumes)": q_try, **rmse_state(o["xf"][:, :2], o["xf"][:, 2:], xv), "loglik": o["loglik"]})
display(pd.DataFrame(rows).set_index("q (filter assumes)").round(4))

# 實驗 3：缺漏的量測（第 80～110 步沒有量測）
y_miss = yv.copy()
y_miss[80:110] = np.nan
m = cv_matrices(Q_TRUE, R_TRUE)
kf_true = KalmanFilter(m["F"], m["H"], m["Q"], m["R"], np.zeros(4), np.eye(4) * 100)
o_miss = kf_true.filter(y_miss)
xs_miss, Ps_miss = kf_true.smooth(o_miss)
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].plot(xv[:, 0], color="k", label="true px")
axes[0].plot(o_miss["xf"][:, 0], label="filter")
axes[0].plot(xs_miss[:, 0], label="smoother")
axes[0].scatter(np.arange(len(y_miss)), y_miss[:, 0], s=4, color="tab:red", alpha=0.5)
axes[0].axvspan(80, 110, color="gray", alpha=0.2)
axes[0].legend()
axes[1].plot(np.sqrt(o_miss["Pf"][:, 0, 0]), label="filter: √P[px]")
axes[1].plot(np.sqrt(Ps_miss[:, 0, 0]), label="smoother: √P[px]")
axes[1].axvspan(80, 110, color="gray", alpha=0.2)
axes[1].legend()
plt.show()

In [ ]:
# TODO 7（問答）TODO 3–6 與檢查
#
# (1) 暴力的高斯條件分布與 smoother、filter 的差異是多少？這說明 Kalman filter 在算什麼？（前置複習 4）
#     暴力法要解一個多大的線性系統？n = 200 步時呢？Kalman filter 每一步只要解多大的？
# (2) 跟 statsmodels 的差異是多少？為什麼要傳 P0 + q 而不是 P0？
# 答：

In [ ]:
# TODO 8（問答）實驗 1–3
#
# (1) 實驗 1：Kalman gain 從第 1 步到第 50 步怎麼變？穩態的值是多少？用穩態 gain 當 α 的 SES，跟 Kalman 濾波差多少？
#     → 第 1 章的 SES 其實是「局部水準模型」的穩態 Kalman filter，α 由 q/r（訊號與雜訊的比例）決定。q/r 大，α 大還是小？
# 答：
#
# (2) 實驗 2：濾波器假設的 q 太小（0.0005）或太大（5）時，位置、速度的 RMSE 怎麼變？log-likelihood 在哪個 q 最大？跟真實的 q = 0.05 一致嗎？
#     q 太小 = 「太相信模型」、q 太大 = 「太相信量測」—— 各會出現什麼樣的錯誤？
# 答：
#
# (3) 實驗 3：缺漏期間，filter 的估計長什麼樣子（直線？）不確定性 √P 怎麼變？smoother 呢？為什麼 smoother 在缺漏的「中間」最不確定？
# 答：

---
## 4. 估計 $q$、$r$：最大概似；對照 statsmodels

真實情況下 $q$、$r$ 不知道。用 3C 的 log-likelihood，對 $(\log q, \log r)$ 做數值最佳化（取 log 保證是正的）。

1. **2D 等速運動**：在 train 軌跡上估計，跟真值（0.05、4）比。
2. **局部水準模型**：手寫 MLE vs `UnobservedComponents(level="llevel").fit()`（statsmodels 用擴散初始化，結果會很接近、不完全一樣）。

In [ ]:
from scipy.optimize import minimize


def fit_qr_cv(Y: np.ndarray) -> tuple[float, float, float]:
    """2D 等速運動：最大化 log-likelihood 估計 (q, r)。回傳 (q, r, loglik)。"""
    def nll(theta: np.ndarray) -> float:
        m = cv_matrices(np.exp(theta[0]), np.exp(theta[1]))
        return -KalmanFilter(m["F"], m["H"], m["Q"], m["R"], np.zeros(4), np.eye(4) * 100).filter(Y)["loglik"]
    res = minimize(nll, x0=[np.log(0.1), np.log(1.0)], method="Nelder-Mead", options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 2000})
    return float(np.exp(res.x[0])), float(np.exp(res.x[1])), float(-res.fun)


t0 = time.perf_counter()
q_hat, r_hat, ll_hat = fit_qr_cv(sim_tr["y"])
print(f"2D：q̂ = {q_hat:.4f}（真值 {Q_TRUE}）、r̂ = {r_hat:.3f}（真值 {R_TRUE}）；log-lik {ll_hat:.2f}；{time.perf_counter() - t0:.1f} 秒")


def fit_qr_ll(y: np.ndarray) -> tuple[float, float]:
    def nll(theta: np.ndarray) -> float:
        return -KalmanFilter(np.eye(1), np.eye(1), np.eye(1) * np.exp(theta[0]), np.eye(1) * np.exp(theta[1]), np.zeros(1), np.eye(1) * 1e4).filter(y[:, None])["loglik"]
    res = minimize(nll, x0=[0.0, 0.0], method="Nelder-Mead", options={"xatol": 1e-7, "fatol": 1e-7})
    return float(np.exp(res.x[0])), float(np.exp(res.x[1]))


q_ll_hat, r_ll_hat = fit_qr_ll(ll_sim["y"])
uc_fit = UnobservedComponents(ll_sim["y"], level="llevel").fit(disp=False)
print(f"局部水準：手寫 q̂ = {q_ll_hat:.4f}、r̂ = {r_ll_hat:.4f}；statsmodels sigma2.level = {uc_fit.params[1]:.4f}、sigma2.irregular = {uc_fit.params[0]:.4f}（真值 q = 0.5、r = 2）")

In [ ]:
# TODO 9（問答）
#
# (1) 2D 的 q̂、r̂ 離真值多遠？哪一個估得比較準？為什麼 r 比較好估？（提示：每一步都有 2 個量測直接告訴你 r；q 只透過「速度的變化」間接看到）
# (2) 局部水準：手寫的 MLE 跟 statsmodels 差多少？兩者的初始化不同（P0 = 1e4 vs 擴散初始化）—— 為什麼影響不大？
# (3) 用 MLE 估計 q、r，跟第 1 章「最小化一步預測誤差的 SSE 估計 α」有什麼關係？（提示：概似裡的 v_t 就是一步預測誤差）
# 答：

---
## 5. 調參：MLE vs 直接在 validation 上調

兩種選 $q$ 的方法（$r$ 固定用 MLE 的 $\hat r$）：

1. **MLE**（第 4 節，只用 train 的量測）；
2. **網格 + validation**：在 validation 軌跡上，選讓**位置 RMSE**（對真相）最小的 $q$。—— 在模擬裡可以這樣做，真實世界沒有真相，只能用 MLE 或「一步預測誤差」。

兩種都只用 train／validation，test 留到第 6 節。

In [ ]:
q_grid = np.logspace(-3, 0, 13)
rows = []
for q_try in q_grid:
    m = cv_matrices(q_try, r_hat)
    o = KalmanFilter(m["F"], m["H"], m["Q"], m["R"], np.zeros(4), np.eye(4) * 100).filter(yv)
    pred_err = yv[1:] - (m["H"] @ o["xp"][1:].T).T
    rows.append({"q": q_try, **rmse_state(o["xf"][:, :2], o["xf"][:, 2:], xv), "val loglik": o["loglik"], "one-step pred RMSE": float(np.sqrt((pred_err[BURN:] ** 2).mean()))})
grid = pd.DataFrame(rows).set_index("q")
display(grid.round(4))
q_val = float(grid["pos RMSE"].idxmin())
q_ll_best = float(grid["val loglik"].idxmax())
print(f"validation 位置 RMSE 最低：q = {q_val:.4f}；validation log-lik 最高：q = {q_ll_best:.4f}；train MLE：q̂ = {q_hat:.4f}")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogx(grid.index, grid["pos RMSE"], "o-", label="pos RMSE (vs truth)")
ax.semilogx(grid.index, grid["vel RMSE"], "o-", label="vel RMSE (vs truth)")
ax.axvline(q_hat, color="gray", ls=":", label="train MLE")
ax.legend()
plt.show()

In [ ]:
# TODO 10（問答）
#
# (1) 位置 RMSE 最低的 q、validation log-likelihood 最高的 q、train MLE 的 q̂，三者接近嗎？
# (2) 「一步預測誤差」（不需要真相）最小的 q 跟「位置 RMSE」（需要真相）最小的 q 一致嗎？真實世界中你會用哪一個選 q？
# (3) RMSE 對 q 的曲線在最低點附近平不平？q 估錯 2 倍會差多少？（robustness）
# 答：

---
## 6. 測試集評估

test 軌跡（seed 2）。比較：原始量測、SES、移動直線、Kalman filter（MLE 的 $\hat q, \hat r$）、Kalman filter（真值 $q, r$，理論上的最好）、RTS smoother（事後分析，用了未來的量測 —— 不能拿來跟即時的方法直接比，放在最後當參考）。成績寫進 `results/kalman_sim.csv`。

In [ ]:
from learning_map.results import compare, log_result

BENCH = "kalman_sim"
yt, xt = sim_te["y"], sim_te["x"]
diff_t = np.vstack([np.full((1, 2), np.nan), np.diff(yt, axis=0)])
m_hat, m_true = cv_matrices(q_hat, r_hat), cv_matrices(Q_TRUE, R_TRUE)
kf_hat = KalmanFilter(m_hat["F"], m_hat["H"], m_hat["Q"], m_hat["R"], np.zeros(4), np.eye(4) * 100)
o_hat = kf_hat.filter(yt)
o_true = KalmanFilter(m_true["F"], m_true["H"], m_true["Q"], m_true["R"], np.zeros(4), np.eye(4) * 100).filter(yt)
xs_hat, Ps_hat = kf_hat.smooth(o_hat)
results = {
    "raw measurement": rmse_state(yt, diff_t, xt),
    "SES α=0.3": rmse_state(np.column_stack([ses(yt[:, d], 0.3) for d in range(2)]), np.column_stack([ses(np.nan_to_num(diff_t[:, d]), 0.3) for d in range(2)]), xt),
    "moving line (5)": rmse_state(*moving_line(yt, 5), xt),
    "Kalman filter (MLE q, r)": rmse_state(o_hat["xf"][:, :2], o_hat["xf"][:, 2:], xt),
    "Kalman filter (true q, r)": rmse_state(o_true["xf"][:, :2], o_true["xf"][:, 2:], xt),
    "RTS smoother (MLE q, r; uses future)": rmse_state(xs_hat[:, :2], xs_hat[:, 2:], xt),
}
test_table = pd.DataFrame(results).T
display(test_table.round(4))
for name, m_ in results.items():
    log_result(BENCH, name, m_, **({"q": round(q_hat, 5), "r": round(r_hat, 4)} if "MLE" in name else {}))
compare(BENCH, "pos RMSE", higher_is_better=False)

---
## 7. 診斷：濾波器「自知之明」嗎？

1. **NIS**（normalized innovation squared）$\mathbf v_t^\top S_t^{-1}\mathbf v_t$：模型對的話 $\sim \chi^2_2$（前置複習 5），平均 2；95% 的值應該落在 $[\chi^2_2(0.025), \chi^2_2(0.975)]$。
2. **NEES**（normalized estimation error squared）$(\mathbf x_t - \hat{\mathbf x}_{t\mid t})^\top P_{t\mid t}^{-1}(\mathbf x_t - \hat{\mathbf x}_{t\mid t})$（需要真相）：平均應該 ≈ 4（狀態維度）。
3. **模型錯了會怎樣**：讓物體在第 100 步突然轉彎（速度方向轉 90°），用同一個濾波器追蹤。

In [ ]:
nis = np.einsum("ti,tij,tj->t", o_hat["v"], np.linalg.inv(o_hat["S"]), o_hat["v"])
err_x = xt - o_hat["xf"]
nees = np.einsum("ti,tij,tj->t", err_x, np.linalg.inv(o_hat["Pf"]), err_x)
lo, hi = stats.chi2.ppf([0.025, 0.975], 2)
print(f"NIS 平均 {nis[BURN:].mean():.3f}（應 ≈ 2）；落在 95% 區間 [{lo:.3f}, {hi:.3f}] 的比例 {np.mean((nis[BURN:] > lo) & (nis[BURN:] < hi)):.3f}；NEES 平均 {nees[BURN:].mean():.3f}（應 ≈ 4）")
inn_acf = [np.corrcoef(o_hat["v"][BURN + k :, 0], o_hat["v"][BURN : len(o_hat["v"]) - k, 0])[0, 1] for k in range(1, 11)]
print("創新（x 方向）的自我相關 lag 1–5：", np.round(inn_acf[:5], 3))

# 突然轉彎
sim_turn = simulate_cv(seed=3)
x_turn = sim_turn["x"].copy()
x_turn[100:, 2:] = x_turn[100:, [3, 2]] * np.array([-1, 1]) * 3  # 速度轉 90° 並變成 3 倍
for t in range(100, N_STEPS):
    x_turn[t, :2] = x_turn[t - 1, :2] + x_turn[t - 1, 2:] * DT
y_turn = x_turn[:, :2] + np.random.default_rng(4).normal(0, np.sqrt(R_TRUE), (N_STEPS, 2))
o_turn = kf_hat.filter(y_turn)
nis_turn = np.einsum("ti,tij,tj->t", o_turn["v"], np.linalg.inv(o_turn["S"]), o_turn["v"])
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].plot(x_turn[:, 0], x_turn[:, 1], color="k", label="true (turns at t = 100)")
axes[0].plot(o_turn["xf"][:, 0], o_turn["xf"][:, 1], label="Kalman filter")
axes[0].scatter(y_turn[:, 0], y_turn[:, 1], s=4, color="tab:red", alpha=0.4)
axes[0].legend()
axes[1].semilogy(nis_turn, lw=0.8)
axes[1].axhline(hi, color="gray", ls=":", label="97.5% χ²₂")
axes[1].axvline(100, color="k", lw=0.5)
axes[1].legend()
axes[1].set_title("NIS (log scale)")
plt.show()
print(f"轉彎後 10 步內的位置 RMSE：{np.sqrt(((o_turn['xf'][100:110, :2] - x_turn[100:110, :2]) ** 2).mean()):.2f}；轉彎前：{np.sqrt(((o_turn['xf'][BURN:100, :2] - x_turn[BURN:100, :2]) ** 2).mean()):.2f}")

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 測試集：Kalman filter（MLE）的位置、速度 RMSE 比原始量測、SES、移動直線好多少？跟用真值 q、r 差多少？smoother 又好多少？
# (2) NIS、NEES 的平均接近理論值嗎？創新有沒有自我相關？這些檢查「不需要真相」的是哪一個？
# (3) 轉彎：NIS 在第 100 步之後怎麼變？濾波器多久才「追上」？你會怎麼讓濾波器對這種情況更穩健？（提示：加大 q、多個模型切換（IMM）、偵測 NIS 異常就重設）
# (4) 本章與前幾章的連結（填表）：
#     |              | 狀態 | F | 本章學到的等價關係 |
#     | 第 1 章 SES  |      |   |                   |
#     | 第 2 章 AR(1)|      |   |                   |
#     | HMM          |      |   |                   |
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

模擬的 2D 等速運動（$q = 0.05$、$r = 4$、200 步；test = seed 2）

| 方法 | 做法 | 用到未來？ | 位置 RMSE | 速度 RMSE |
| --- | --- | --- | --- | --- |
| 原始量測 | | 否 | | |
| SES、移動直線 | | 否 | | |
| Kalman filter（MLE $q, r$） | | 否 | | |
| Kalman filter（真值） | | 否 | | |
| RTS smoother | | 是 | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ Kalman filter 的推導（最小變異數線性估計；貝氏遞迴）；Joseph form 的數值穩定性
- ☐ Riccati 方程式與穩態 gain；可觀測性、可控制性
- ☐ ARIMA、指數平滑、結構時間序列模型的狀態空間形式（statsmodels `SARIMAX` 內部就是 Kalman filter）

**延伸**
- ☐ Extended KF（非線性：線性化）、Unscented KF（sigma points）、粒子濾波
- ☐ 用 EM 估計 $Q$、$R$（Shumway & Stoffer）
- ☐ 多目標追蹤、IMM（交互式多模型）